In [1]:
# Si estás en Google Colab o en un entorno donde PennyLane no está instalado,

!pip install pennylane

# Sesión 11 Laboratorio de Computación Cuántica 2

## Introducción a los Algoritmos Cuanticos Variacionales: El Variational Quantum Eigensolver (VQE)

Hoy vamos a introducirnos al area de los algoritmos cuánticos Variacionales. Estos algoritmos se programaron con la intención de vencer los problemas que las computadoras NISQ presentan

### Introducción a los algoritmos cuánticos variacionales

Los **algoritmos cuánticos variacionales** son algoritmos híbridos cuántico-clásicos.

La idea general es usar dos partes:

1. Una **computadora cuántica**, o simulador cuántico, que prepara estados cuánticos parametrizados.
2. Una **computadora clásica**, que ajusta los parámetros del circuito para optimizar alguna función de costo.

En estos algoritmos, no diseñamos directamente un circuito cuántico fijo que resuelva el problema. En cambio, proponemos una familia de circuitos dependientes de parámetros:

$$
|\psi(\theta)\rangle = U(\theta)|0\rangle
$$

donde $\theta$ representa uno o varios parámetros ajustables.

Después, medimos una cantidad física o matemática de interés y usamos un optimizador clásico para actualizar los parámetros:

$$
\theta \longrightarrow \theta'
$$

El objetivo es encontrar los parámetros que minimizan o maximizan una función de costo.

Estos algoritmos son importantes porque están pensados para computadoras cuánticas ruidosas de escala intermedia, conocidas como dispositivos NISQ.

### ¿Qué es un Hamiltoniano?

En mecánica cuántica, el Hamiltoniano es el operador que representa la energía de un sistema físico.

Se suele denotar como:

$$
H
$$

Si un sistema cuántico está en el estado $|\psi\rangle$, entonces la energía esperada de ese estado es:

$$
E(\psi) = \langle \psi | H | \psi \rangle
$$

El Hamiltoniano contiene la información del sistema que queremos estudiar. Por ejemplo, puede representar:

- La energía de una molécula.
- La interacción entre espines.
- Un sistema de partículas.
- Un problema de optimización escrito en lenguaje cuántico.

En computación cuántica, normalmente escribimos el Hamiltoniano como una combinación de operadores de Pauli:

$$
H = c_0 I + c_1 X + c_2 Y + c_3 Z + \cdots
$$

donde $X$, $Y$ y $Z$ son las matrices de Pauli.

### ¿Qué es el estado base?

Los estados propios de un Hamiltoniano satisfacen:

$$
H |E_i\rangle = E_i |E_i\rangle
$$

donde:

- $|E_i\rangle$ es un estado propio del Hamiltoniano.
- $E_i$ es la energía asociada a ese estado.

El **estado base** es el estado de menor energía posible del sistema. Es decir, si las energías del Hamiltoniano son:

$$
E_0 \leq E_1 \leq E_2 \leq \cdots
$$

entonces $E_0$ es la **energía del estado base**, y $|E_0\rangle$ es el **estado base**.

Encontrar el estado base es un problema muy importante en física, química cuántica y optimización.
]
Por ejemplo, en química cuántica, conocer la energía del estado base de una molécula ayuda a entender su estabilidad y sus propiedades físicas.

### ¿Qué hace el VQE?

El **Variational Quantum Eigensolver**, o **VQE**, es un algoritmo cuántico variacional diseñado para aproximar la energía del estado base de un Hamiltoniano.

La idea central es usar el principio variacional:

$$
\langle \psi(\theta) | H | \psi(\theta) \rangle \geq E_0
$$

para cualquier estado normalizado $|\psi(\theta)\rangle$.

Esto significa que la energía esperada de cualquier estado de prueba siempre está por encima o igual a la energía real del estado base.

Entonces, si logramos minimizar

$$
E(\theta) = \langle \psi(\theta) | H | \psi(\theta) \rangle
$$

podemos aproximarnos a la energía del estado base:

$$
E(\theta_{\text{opt}}) \approx E_0
$$

El ciclo del VQE es:

1. Elegir un Hamiltoniano $H$.
2. Proponer un circuito parametrizado, llamado ansatz.
3. Preparar el estado $|\psi(\theta)\rangle$.
4. Medir la energía esperada:

$$
E(\theta) = \langle \psi(\theta) | H | \psi(\theta) \rangle
$$

5. Usar un optimizador clásico para actualizar $\theta$.
6. Repetir hasta que la energía deje de disminuir.

### ¿Qué es un ansatz?

Un **ansatz** es una propuesta de forma para la solución.

En VQE, el ansatz es un circuito cuántico parametrizado que prepara una familia de estados:

$$
|\psi(\theta)\rangle = U(\theta)|0\rangle
$$

No conocemos de antemano el estado base exacto, así que usamos el ansatz para explorar una región del espacio de estados.

Un buen ansatz debe tener dos propiedades importantes:

1. Debe ser suficientemente expresivo para aproximar el estado base.
2. Debe ser suficientemente simple para poder implementarse en una computadora cuántica real.

Si el ansatz es demasiado simple, quizá no pueda representar el estado base.

Si el ansatz es demasiado complicado, puede ser difícil de optimizar o implementar en hardware cuántico real.

La mejor forma de aplicar lo aprendido, es a un problema conocido relativamente sencillo: Estado base de un Hamiltoniano de un qubit

### Ejemplo simple: Hamiltoniano de un qubit

Para este notebook usaremos un Hamiltoniano muy simple:

$$
H = X + Z
$$

donde $X$ y $Z$ son matrices de Pauli.

Nuestro objetivo será encontrar aproximadamente la menor energía posible de este Hamiltoniano usando VQE.

Como el sistema tiene un solo qubit, podemos usar un ansatz muy sencillo:

$$
|\psi(\theta)\rangle = R_y(\theta)|0\rangle
$$

La compuerta $R_y(\theta)$ rota el estado inicial alrededor del eje $y$ de la esfera de Bloch.

Para este ansatz se cumple que:

$$
\langle Z \rangle = \cos(\theta)
$$

y

$$
\langle X \rangle = \sin(\theta)
$$

Por lo tanto, la energía esperada es:

$$
E(\theta) = \langle X + Z \rangle
$$

es decir:

$$
E(\theta) = \sin(\theta) + \cos(\theta)
$$

El VQE intentará encontrar el valor de $\theta$ que minimiza esta energía.

In [2]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Crear un dispositivo de simulación de un qubit
dev = qml.device("default.qubit", wires=1)

In [4]:
# Hamiltoniano:
# H = X + Z

coeffs = [1.0, 1.0]
observables = [qml.PauliX(0), qml.PauliZ(0)]

H = qml.Hamiltonian(coeffs, observables)

print(H)

1.0 * X(0) + 1.0 * Z(0)


### Circuito variacional

Ahora definimos el ansatz:

$$
|\psi(\theta)\rangle = R_y(\theta)|0\rangle
$$

Este circuito depende de un parámetro entrenable $\theta$.

Después medimos la energía esperada:

$$
E(\theta) = \langle \psi(\theta) | H | \psi(\theta) \rangle
$$

En PennyLane, esto se hace usando `qml.expval(H)`.

In [5]:
@qml.qnode(dev)
def circuit(theta):
    qml.RY(theta, wires=0)
    return qml.expval(H)

In [17]:
theta_test = np.array(0.0, requires_grad=True)

energy = circuit(theta_test)

print("Energía para theta = 0:", energy)

Energía para theta = 0: 1.0


### Función de costo

En VQE, la función de costo es la energía esperada del Hamiltoniano:

$$
C(\theta) = E(\theta)
$$

donde:

$$
E(\theta) = \langle \psi(\theta) | H | \psi(\theta) \rangle
$$

El optimizador clásico intentará encontrar:

$$
\theta_{\text{opt}} = \arg\min_{\theta} E(\theta)
$$

Es decir, buscara minimizar la función de costo

In [18]:
def cost(theta):
    return circuit(theta)

In [21]:
# Damos un parametro inicial
theta = np.array(0.1, requires_grad=True)

# Optimizador clásico
opt = qml.GradientDescentOptimizer(stepsize=0.2)

# Número de iteraciones de optimización
max_iterations = 80

# Guardamos la historia de energías
energy_history = []

for n in range(max_iterations):
    theta, energy = opt.step_and_cost(cost, theta) # Actualizamos theta y obtenemos la energía
    energy_history.append(energy) # Guardamos la energía en cada iteración

    if n % 10 == 0:
        print(f"Iteración {n:3d} | Energía = {energy:.8f} | theta = {theta:.8f}")  # Imprime cada 10 iteraciones

print("\nResultado final:")
print("theta óptimo:", theta)
print("energía mínima aproximada:", cost(theta))

Iteración   0 | Energía = 1.09483758 | theta = -0.07903415
Iteración  10 | Energía = -1.35282341 | theta = -2.14289807
Iteración  20 | Energía = -1.41413156 | theta = -2.34847119
Iteración  30 | Energía = -1.41421346 | theta = -2.35591655
Iteración  40 | Energía = -1.41421356 | theta = -2.35618449
Iteración  50 | Energía = -1.41421356 | theta = -2.35619413
Iteración  60 | Energía = -1.41421356 | theta = -2.35619448
Iteración  70 | Energía = -1.41421356 | theta = -2.35619449

Resultado final:
theta óptimo: -2.3561944901689533
energía mínima aproximada: -1.414213562373095


### Interpretación del resultado

El VQE encontró un valor de $\theta$ que minimiza la energía esperada:

$$
E(\theta) = \langle \psi(\theta) | H | \psi(\theta) \rangle
$$

En este ejemplo, el Hamiltoniano era:

$$
H = X + Z
$$

y la energía exacta del estado base es:

$$
E_0 = -\sqrt{2}
$$

El algoritmo no diagonalizó directamente la matriz del Hamiltoniano. En cambio, hizo lo siguiente:

1. Preparó un estado de prueba usando un circuito parametrizado.
2. Midió la energía esperada del Hamiltoniano.
3. Ajustó el parámetro $\theta$ usando un optimizador clásico.
4. Repitió el proceso hasta encontrar una energía mínima.

Este es el corazón del VQE: usar un circuito cuántico para evaluar energías y una computadora clásica para optimizar los parámetros.

Pasemos a un ejemplo algo mas complicado: VQE en 2 Qubits:

### VQE con 2 qubits

Ahora vamos a construir un ejemplo ligeramente más interesante usando 2 qubits.

Usaremos el Hamiltoniano:

$$
H = Z_0 + Z_1 + X_0X_1
$$

Este Hamiltoniano tiene tres términos:

$$
Z_0
$$

$$
Z_1
$$

$$
X_0X_1
$$

Los primeros dos términos miden contribuciones locales de cada qubit. El tercer término acopla ambos qubits.

Nuestro objetivo es encontrar una aproximación a la energía del estado base:

$$
E_0 = \min_{\psi} \langle \psi | H | \psi \rangle
$$

Para esto usaremos VQE.

## Ansatz de 2 qubits

Para este ejemplo usaremos un ansatz simple:

$$
|\psi(\theta_0,\theta_1)\rangle =
\text{CNOT}_{0,1}
\left(
R_y(\theta_0) \otimes R_y(\theta_1)
\right)
|00\rangle
$$

Primero aplicamos rotaciones en ambos qubits:

$$
R_y(\theta_0)
$$

y

$$
R_y(\theta_1)
$$

Luego aplicamos una compuerta CNOT para introducir entrelazamiento entre los qubits.

Esto es importante porque muchos estados base de Hamiltonianos de varios qubits no pueden representarse bien usando solamente estados producto.

La CNOT permite que el ansatz explore estados entrelazados.

In [22]:
# Dispositivo de 2 qubits
dev_2q = qml.device("default.qubit", wires=2)

In [ ]:
# Hamiltoniano:
# H = Z_0 + Z_1 + X_0 X_1

coeffs_2q = [1.0, 1.0, 1.0]

observables_2q = [
    qml.PauliZ(0),
    qml.PauliZ(1),
    qml.PauliX(0) @ qml.PauliX(1)
]

H_2q = qml.Hamiltonian(coeffs_2q, observables_2q)

print(H_2q)

1.0 * Z(0) + 1.0 * Z(1) + 1.0 * (X(0) @ X(1))


### Circuito variacional

Ahora definimos el circuito variacional.

El circuito depende de dos parámetros:

$$
\theta =
(\theta_0,\theta_1)
$$

El estado preparado por el circuito es:

$$
|\psi(\theta_0,\theta_1)\rangle
$$

Después calculamos la energía esperada:

$$
E(\theta_0,\theta_1)
=
\langle \psi(\theta_0,\theta_1) |
H
| \psi(\theta_0,\theta_1) \rangle
$$

In [23]:
@qml.qnode(dev_2q)
def circuit_2q(theta):
    # Rotaciones parametrizadas
    qml.RY(theta[0], wires=0)
    qml.RY(theta[1], wires=1)

    # Entrelazamiento
    qml.CNOT(wires=[0, 1])

    # Medimos el valor esperado del Hamiltoniano
    return qml.expval(H_2q)

In [24]:
# Probamos el circuito con parámetros iniciales
theta_test = np.array([0.1, 0.2], requires_grad=True)

energy_test = circuit_2q(theta_test)

print("Energía inicial:", energy_test)

Energía inicial: 2.07000790912667


### Función de costo

La función de costo del VQE será la energía esperada:

$$
C(\theta_0,\theta_1)
=
E(\theta_0,\theta_1)
$$

Queremos encontrar los parámetros que minimicen esta energía:

$$
(\theta_0^\star,\theta_1^\star)
=
\arg\min_{\theta_0,\theta_1}
E(\theta_0,\theta_1)
$$

In [13]:
def cost_2q(theta):
    return circuit_2q(theta)

In [25]:
# Parámetros iniciales
theta = np.array([0.1, 0.2], requires_grad=True)

# Optimizador clásico
opt = qml.GradientDescentOptimizer(stepsize=0.2)

# Iteraciones
max_iterations = 100

# Guardamos la historia de energías
energy_history_2q = []

for n in range(max_iterations):
    theta, energy = opt.step_and_cost(cost_2q, theta)
    energy_history_2q.append(energy)

    if n % 10 == 0:
        print(f"Iteración {n:3d} | Energía = {energy:.8f} | theta = {theta}")

print("\nResultado final:")
print("theta óptimo:", theta)
print("energía mínima aproximada:", cost_2q(theta))

Iteración   0 | Energía = 2.07000791 | theta = [-0.05946547  0.23953536]
Iteración  10 | Energía = -2.18706785 | theta = [-2.60898016  0.23832517]
Iteración  20 | Energía = -2.23529800 | theta = [-2.67738889  0.0340677 ]
Iteración  30 | Energía = -2.23605302 | theta = [-2.6779364   0.00474799]
Iteración  40 | Energía = -2.23606769 | theta = [-2.67794488e+00  6.61530380e-04]
Iteración  50 | Energía = -2.23606797 | theta = [-2.67794504e+00  9.21695311e-05]
Iteración  60 | Energía = -2.23606798 | theta = [-2.67794504e+00  1.28417710e-05]
Iteración  70 | Energía = -2.23606798 | theta = [-2.67794504e+00  1.78921471e-06]
Iteración  80 | Energía = -2.23606798 | theta = [-2.67794504e+00  2.49287212e-07]
Iteración  90 | Energía = -2.23606798 | theta = [-2.67794504e+00  3.47326197e-08]

Resultado final:
theta óptimo: [-2.67794504e+00  5.89347339e-09]
energía mínima aproximada: -2.23606797749979


### Comparación con la solución exacta

Como este sistema tiene solamente 2 qubits, podemos calcular la solución exacta diagonalizando la matriz del Hamiltoniano.

El espacio de Hilbert tiene dimensión:

$$
2^2 = 4
$$

Entonces el Hamiltoniano se puede representar como una matriz de $4 \times 4$.

La base computacional es:

$$
|00\rangle,\ |01\rangle,\ |10\rangle,\ |11\rangle
$$

Compararemos la energía obtenida por VQE con el eigenvalor más pequeño de la matriz de $H$.

In [16]:
# Matrices de Pauli
I = np.eye(2)
X = np.array([[0, 1], [1, 0]])
Z = np.array([[1, 0], [0, -1]])

# Productos tensoriales
Z0 = np.kron(Z, I)
Z1 = np.kron(I, Z)
X0X1 = np.kron(X, X)

# Hamiltoniano matricial
H_matrix_2q = Z0 + Z1 + X0X1

# Diagonalización exacta
eigenvalues_2q, eigenvectors_2q = np.linalg.eigh(H_matrix_2q)

print("Eigenvalores exactos:")
print(eigenvalues_2q)

print("\nEnergía exacta del estado base:")
print(eigenvalues_2q[0])

print("\nEnergía obtenida por VQE:")
print(cost_2q(theta))



Eigenvalores exactos:
[-2.23606798 -1.          1.          2.23606798]

Energía exacta del estado base:
-2.23606797749979

Energía obtenida por VQE:
-2.23606797749979
